# Value Strategy — intangible-adjusted B/M, quality, momentum, ML regime detection

**Narrative** notebook: it imports the logic from the `value_strategy` package (under `src/`) 
and walks through the strategy part by part. All the heavy code lives in the modules; this 
notebook only orchestrates and comments.

**Strategy**: US small/mid cap long/short equity. Value signal = intangible-adjusted B/M 
(KC + OC, Peters & Taylor 2017), ranked by sector, filtered by quality + 6M momentum. Short = 
bottom quality of the growth bucket. ML detection of *junk rally* regimes to dynamically reduce 
the short.

> **Data**: CRSP + Compustat via WRDS, IS 2003-2013, OOS 2014-2024.  
> First run: WRDS connection (caches the data). Afterwards: `USE_CACHE = True`.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Make the package importable from the notebook (src/ on the path)
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from value_strategy import config, signals, factors, dynamic_short, plots, reports
from value_strategy import portfolio as pf
from value_strategy.data import build as data_build
from value_strategy.ml_regime import pipeline as ml_pipeline

config.ensure_dirs()
USE_CACHE = True   # replay from the local cache data/cache/ (fast, no WRDS); set to False to start from WRDS

## Part 1 — Data preparation

WRDS connection, CRSP + Compustat extraction, survivorship-bias correction (Shumway 2001), 
intangible capital KC/OC computation, cleaning, Amihud illiquidity and dynamic costs. The final 
monthly panel is cached (parquet) for reproducibility.

In [2]:
if USE_CACHE:
    panel, df_comp, crsp_ml, macro = data_build.load_cached_panel()
else:
    panel, df_comp, crsp_ml, macro = data_build.build_panel_from_wrds()
panel.shape

  PART 1 — READING PANEL (cache)


  -> cache read: data/cache/panel.parquet (1,191,883 rows)
  -> cache read: data/cache/compustat.parquet (210,222 rows)
  -> cache read: data/cache/crsp_ml.parquet (1,294,973 rows)


  -> cache read: data/cache/macro.parquet (300 rows)


(1191883, 50)

## Part 2 — In-sample: construction and calibration (2003-2013)

Value signals (sector-neutralized) + quality score + momentum, then long/short portfolio 
construction (semi-annual rebalancing) and performance net of all costs.

In [3]:
panel_sm_is, filtered_is = signals.build_signals(panel, config.IS_START, config.IS_END)
long_is, short_is, _ = pf.construct_portfolios(filtered_is, config.IS_START, config.IS_END)
perf_is, costs_is = pf.compute_performance(panel_sm_is, long_is, short_is, config.IS_START, config.IS_END)

Panel 2003-2013: 549,723 obs — 7,063 stocks


Investable universe: 4,196 unique stocks


Filtered universe: 3,359 stocks


Number of rebalancing dates: 22
Average LONG size:  84.0
Average SHORT size: 130.8



Performance computed: 120 months
Transaction cost: 0.27%/yr
Borrow cost     : 1.26%/yr
Total cost      : 1.53%/yr


## Part 3 — Out-of-sample: application and comparison (2014-2024)

**Same parameters, no recalibration.** We download the Fama-French factors and compare IS vs 
OOS (Sharpe, FF4 alpha, information ratio vs HML).

In [4]:
panel_sm_oos, filtered_oos = signals.build_signals(panel, config.OOS_START, config.OOS_END)
long_oos, short_oos, _ = pf.construct_portfolios(filtered_oos, config.OOS_START, config.OOS_END)
perf_oos, costs_oos = pf.compute_performance(panel_sm_oos, long_oos, short_oos, config.OOS_START, config.OOS_END)

ff5, mom_ff = factors.load_ff_factors()
stats_is, stats_oos = reports.print_is_oos_comparison(perf_is, perf_oos, ff5, mom_ff, costs_is, costs_oos)

Panel 2014-2024: 446,929 obs — 5,724 stocks


Investable universe: 3,964 unique stocks


Filtered universe: 2,976 stocks


Number of rebalancing dates: 22
Average LONG size:  81.3
Average SHORT size: 102.0



Performance computed: 120 months
Transaction cost: 0.34%/yr
Borrow cost     : 0.86%/yr
Total cost      : 1.20%/yr


FF factors loaded successfully

IS vs OOS COMPARISON — Net strategy (after all costs)

Metric                         IS (2003-13)   OOS (2014-24)
-----------------------------------------------------------
Annual return                        12.01%          16.01%
Volatility                           10.66%          11.30%
Sharpe Ratio                           0.92            1.18
Max Drawdown                        -11.99%         -12.83%
Calmar Ratio                           1.00            1.25
Skewness                               0.94            0.78
VaR 5%                               -3.17%          -3.18%
-----------------------------------------------------------
FF4 alpha (annual)                   12.00%          15.71%
FF4 alpha t-stat                       3.52            4.22
FF4 R-squared                         0.036           0.026
Info. Ratio vs HML                     1.09            1.34

  -> Sharpe change IS->OOS: +0.25
  -> IS cost: 1.53%/yr | OOS cost: 1.2

## Part 4 — ML detection of the 'junk rally' regime

Feature engineering (CRSP + Compustat + FRED) -> HMM/GMM labeling of euphoria regimes -> 
supervised walk-forward prediction -> SHORT_REDUCE / FULL_SHORT signals.

In [5]:
ml = ml_pipeline.run_ml_regime(crsp_ml, df_comp, macro)
ml['metrics_df']

  PART 4 — REGIME DETECTION (ML)

[STEP 1] Feature engineering V3...
  Dropped 6,407 rows without ret


  Raw market panel: 300 months (2000-01 -> 2024-12)
  Merged 6 macro series
  Final market panel: 300 months | Features: 56
[STEP 2] Labeling EUPHORIA regimes V3 (min_dur=3, bridge=2)...


  ! Degenerate clustering (minority regime 6.0% < 20%) -> stress-score fallback (median)
  Method: GaussianHMM + stress-score fallback (degenerate clustering)
  Normal   (0):  131 months (43.7%)
  Euphoria (1):  169 months (56.3%)
  Features: ['vix_z', 'rvol_mkt_z', 'ret_dispersion_z', 'credit_spread_z']
  Mean VIX in Normal  : -0.82
  Mean VIX in Euphoria: +0.53
[STEP 3a] Preparing supervised data...
  Samples: 299 | Stress: 56.2% | Period: 2000-01 -> 2024-11
[STEP 3b] Walk-forward V3 (min_train=120, test=12, n=299)...


  Folds: 14 | OOS: 168 | Avg threshold: 0.472

  MODEL COMPARISON — Out-of-Sample V3

  Logistic Regression:
    Accuracy: 0.774 | Precision: 0.840 | Recall: 0.792
    AUC: 0.784 | Brier: 0.1960
    Confusion matrix: [[46, 16], [22, 84]]

  Gradient Boosting:
    Accuracy: 0.774 | Precision: 0.840 | Recall: 0.792
    AUC: 0.821 | Brier: 0.1751
    Confusion matrix: [[46, 16], [22, 84]]

  * Best (AUC): Gradient Boosting (0.821)

[STEP 4b] Feature importance (permutation)...


  Top 10 features:
    vix_z                               0.0369 +/- 0.0102
    turnover_mkt_z                      0.0176 +/- 0.0072
    leverage_mkt_z                      0.0169 +/- 0.0078
    ret_dispersion_z                    0.0129 +/- 0.0080
    log_dollar_vol_z                    0.0107 +/- 0.0075
    vix_z_ma3                           0.0097 +/- 0.0051
    consumer_sent_z_chg3                0.0086 +/- 0.0055
    ted_spread_z_ma3                    0.0075 +/- 0.0033
    fed_funds_z_chg1                    0.0074 +/- 0.0052
    mkt_vol_3m                          0.0062 +/- 0.0021
[STEP 6] Euphoria signals (threshold=0.472)...
  SHORT_REDUCE: 101 | FULL_SHORT: 67
  Euphoria detected: 85/106 (80.2%)
  Overall GB signal accuracy: 78.0%

  PART 4 SUMMARY — EUPHORIA DETECTION
  Regime method  : GaussianHMM + stress-score fallback (degenerate clustering) + smoothing
  Total features : 56
  ML panel       : 300 months
  Avg threshold  : 0.472


,Accuracy,Precision (stress),Recall (stress),F1 (stress),AUC-ROC,Brier Score
Model,,,,,,
Logistic Regression,0.77381,0.84,0.792453,0.815534,0.784236,0.196038
Gradient Boosting,0.77381,0.84,0.792453,0.815534,0.820907,0.175073


## Part 5 — Dynamic short reduction during junk rallies

When the ML detects euphoria, the short weight goes from 100% to 50%. We compare the original 
strategy with the *dynamic short* version (Sharpe, MaxDD, FF4 alpha, costs).

In [6]:
ff = factors.download_ff_factors()
perf_h, summary = dynamic_short.build_dynamic_short(perf_oos, ml['signals_df'], costs_oos)
perf_h, metrics = dynamic_short.attach_ff_and_metrics(perf_h, ff)
ff4_results = dynamic_short.ff4_regression(perf_h, ff)
reports.print_dynamic_short_verdict(perf_h, metrics, ff4_results, summary, costs_oos)

  PART 5 — ML-DRIVEN DYNAMIC SHORT
[5.1b] Aligning ML signals -> OOS returns...

  Period: 2014-12 -> 2024-11 (120 months)
  SHORT_REDUCE months (50%): 85 (71%)
  FULL_SHORT months (100%) : 35 (29%)
  Transitions              : 14
  Total transition cost    : 2.10%

  FINAL SUMMARY — SHOULD WE KEEP THE DYNAMIC SHORT?
  Sharpe : 1.177 -> 1.201 (+0.024)
  CAGR   : 15.3% -> 19.7% (+4.4%)
  MaxDD  : -12.8% -> -11.2% (+1.6%)
  FF4 a  : 0.1571 -> 0.2060
  Short-reduced months (50%): 85/120 (71%) | Transitions: 14
  Annualized cost: 1.20% -> 1.04%

  VERDICT: Sharpe IMPROVED AND MaxDD reduced -> KEEP


## Figures

All figures are saved to `results/charts/`.

In [7]:
plots.setup_style()
plots.plot_cumulative_wealth(perf_is, perf_oos, ff5)
plots.plot_rolling_sharpe(perf_is, perf_oos, ff5)
plots.plot_annual_returns(perf_is, perf_oos, ff5)
plots.plot_drawdown(perf_is, perf_oos, ff5)
plots.plot_calendar_heatmap(perf_oos)
plots.plot_robustness(perf_is, perf_oos, ff5)
plots.plot_ml_dashboard(ml['mkt_df'], ml['results_df'], ml['importance_df'], ml['metrics_df'], ml['regime_method'], ml['avg_threshold'])
plots.plot_dynamic_short(perf_h, metrics, ff4_results, summary)
plots.plot_final_comparison(perf_is, perf_oos, perf_h, ff5, stats_is, stats_oos, metrics['adj'])

  figure: results/charts/fig1_cumulative_wealth.png
  figure: results/charts/fig2_rolling_sharpe.png


  figure: results/charts/fig3_annual_returns.png
  figure: results/charts/fig4_drawdown.png


  figure: results/charts/fig5_calendar_heatmap.png
  figure: results/charts/fig6_robustesse.png


  figure: results/charts/liquidity_regime_dashboard_v3.png


  figure: results/charts/short_dynamique_v3.png


  figure: results/charts/fig_final_cumulative_comparison.png
  Final $1 value: original $12.22 | dyn. short $17.71 | HML $0.82


PosixPath('/home/lucad/Documents/Projets Programmation/Value-Strategy/results/charts/fig_final_cumulative_comparison.png')